<a href="https://colab.research.google.com/github/Nanayeb34/IndabaX-2026/blob/main/Ethical%20Computer%20Vision/notebook/Part1.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Building Ethical Vision Systems
## Part 1 — Computer Vision Foundations Through the Lens of an Audit

**Ghana Data Science Summit 2026 · Ho, Ghana** | *Tutorial lead: Nana Sam Yeboah*

---

A health startup in Accra has built a skin condition detection app for community health workers in rural Ghana. Before it goes live, we have been asked to audit it.


**How to use this notebook:**
1. Run the setup cells below once (they install dependencies and download sample images)
2. Work through each section top to bottom — all code is pre-run with outputs visible
3. Modify any `🧪 EXPERIMENT` cell and re-run to explore
4. Follow the further reading links in each section to go deeper on any concept


---
##  SECTION 00 — SETUP
*Run these cells once. Everything else will work.*

---

In [ ]:
# Install dependencies — takes 2–4 minutes on a fresh Colab instance
%pip install -q torch torchvision timm opencv-python-headless pillow \
             numpy matplotlib seaborn ultralytics transformers scipy gdown
print("✓ Dependencies ready")

In [ ]:
# Clone the repo to get the sample images (shallow clone — fast)
import os
if not os.path.exists("IndabaX-2026"):
    !git clone https://github.com/Nanayeb34/IndabaX-2026.git IndabaX-2026 --depth 1 -q
    print("✓ Repo cloned")
else:
    print("✓ Repo already present")

DATA_DIR = "IndabaX-2026/Ethical Computer Vision/data/sample_images"
print(f"Sample images in: {DATA_DIR}")

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from pathlib import Path
import torch
import torch.nn as nn
import torchvision.transforms as T

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED   = 42
torch.manual_seed(SEED); np.random.seed(SEED)

# ── Inline helper functions (no external utils needed) ──────────────────────

def load_image(path, colour_space="rgb"):
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(f"Cannot load: {path}")
    cs = colour_space.lower()
    if cs == "rgb":  return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if cs == "gray": return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    if cs == "hsv":  return cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    return img  # bgr

def clahe(img, clip=2.0, tile=8):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    eq = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile)).apply(l)
    return cv2.cvtColor(cv2.merge([eq, a, b]), cv2.COLOR_LAB2RGB)

def remove_shadow(img, k=7, blur=21):
    kernel = np.ones((k, k), np.uint8)
    chs = []
    for ch in cv2.split(img):
        bg = cv2.medianBlur(cv2.dilate(ch, kernel), blur)
        chs.append(np.clip(
            cv2.divide(ch.astype(np.float32), bg.astype(np.float32), scale=255),
            0, 255).astype(np.uint8))
    return cv2.merge(chs)

def unsharp(img, alpha=0.5, sigma=3.0):
    blurred = cv2.GaussianBlur(img, (0, 0), sigmaX=sigma)
    return np.clip(cv2.addWeighted(img, 1+alpha, blurred, -alpha, 0), 0, 255).astype(np.uint8)

def canny(img, lo=50, hi=150):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) if img.ndim == 3 else img
    return cv2.Canny(gray, lo, hi)

def to_tensor(pil_img, device="cpu"):
    tfm = T.Compose([T.Resize((224,224)), T.ToTensor(),
                     T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    return tfm(pil_img).unsqueeze(0).to(device)

def predict_topk(model, tensor, names, k=5):
    with torch.no_grad():
        probs = torch.softmax(model(tensor), 1)[0].cpu().numpy()
    idx = probs.argsort()[::-1][:k]
    return [(names[i], float(probs[i])) for i in idx]

def gradcam(model, tensor, target_cls, layer="blocks.5"):
    model.eval(); acts, grads = [], []
    mod = model
    for a in layer.split("."): mod = getattr(mod, a)
    fh = mod.register_forward_hook(lambda m,i,o: acts.append(o.detach()))
    bh = mod.register_full_backward_hook(lambda m,gi,go: grads.append(go[0].detach()))
    out = model(tensor.float().requires_grad_(True)); model.zero_grad()
    out[0, target_cls].backward()
    fh.remove(); bh.remove()
    if not acts or not grads: return np.zeros((7,7))
    a = acts[0].cpu().numpy()[0]; g = grads[0].cpu().numpy()[0]
    w = g.mean(axis=(1,2))
    cam = np.maximum(np.sum(w[:,None,None]*a, axis=0), 0)
    return (cam / (cam.max() + 1e-8)).astype(np.float32)

def clip_classify(model, processor, img_pil, prompts, device="cpu"):
    inp = processor(text=prompts, images=img_pil, return_tensors="pt", padding=True)
    inp = {k: v.to(device) for k, v in inp.items()}
    with torch.no_grad():
        probs = model(**inp).logits_per_image.softmax(1)[0].cpu().numpy()
    return dict(zip(prompts, probs.tolist()))

print(f"✓ All imports and helpers ready | device = {DEVICE}")

---
##  SECTION 01 — THE IMAGE ARRIVES

---

A community health worker in Brong-Ahafo has just sent her first image. She photographed the patient's arm outdoors, midday sun, with her Tecno Spark Android. The image is in our hands now. Before any model sees it, we need to understand what it actually is — not in clinical terms, but in mathematical ones.


In [ ]:
SAMPLE = f"{DATA_DIR}/prurigo_modularis.jpg"
img    = load_image(SAMPLE)          # RGB uint8 numpy array
img_g  = load_image(SAMPLE, "gray")
img_h  = load_image(SAMPLE, "hsv")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, im, title, cmap in zip(axes,
    [img, cv2.cvtColor(load_image(SAMPLE,"bgr"), cv2.COLOR_BGR2RGB),
     img_g, img_h],
    ["RGB (correct)", "BGR displayed as RGB(wrong colours)",
     "Grayscale", "HSV (raw display)"],
    [None, None, "gray", None]):
    ax.imshow(im, cmap=cmap); ax.set_title(title, fontsize=10); ax.axis("off")
plt.suptitle("Same image — four colour representations", fontsize=12)
plt.tight_layout(); plt.show()
print(f"Shape: {img.shape}  dtype: {img.dtype}  value range: [{img.min()}, {img.max()}]")

### Digital Images as Tensors

A digital image is a three-dimensional array of numbers:

$$I \in \mathbb{R}^{H \times W \times C}$$

where $H$ = height, $W$ = width, $C$ = channels (3 for RGB, 1 for grayscale). Every element is an integer from 0 to 255. The model sees only these numbers — no colours, no shapes, no clinical context.

$$\text{pixel}_{(i,j)} = [R_{(i,j)},\; G_{(i,j)},\; B_{(i,j)}] \in [0,\,255]^3$$

**RGB vs BGR.** OpenCV defaults to BGR (Blue-Green-Red). Display libraries expect RGB. This mismatch is one of the most common sources of bugs in CV code.

**Colour spaces for skin imaging:**
- **RGB** — standard for display
- **HSV (Hue·Saturation·Value)** — separates *colour* (H) from *brightness* (V). The H channel is nearly constant under lighting changes — critical for outdoor phone photos
- **Grayscale** — single channel; loses colour but works well for edge detection


In [ ]:
# HSV decomposition — the V channel reveals the lighting problem
h, s, v = cv2.split(img_h)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, ch, title, cmap in zip(axes,
    [img, h, s, v],
    ["Original RGB", "H — Hue (colour type, stable in light)",
    "S — Saturation (colour intensity)", "V — Value (brightness — the problem)"],
    [None, "hsv", "gray", "gray"]):
    ax.imshow(ch, cmap=cmap); ax.set_title(title, fontsize=10); ax.axis("off")
plt.suptitle("HSV decomposition — the V channel shows strong lighting variation across the lesion",
             fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# Intensity histogram — the lighting problem made visible
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, ch, name, colour in zip(axes, [0, 1, 2], ["Red", "Green", "Blue"],
                                 ["#e74c3c", "#27ae60", "#2980b9"]):
    vals = img[:,:,ch].ravel()
    ax.hist(vals, bins=64, color=colour, alpha=0.8, edgecolor="none")
    ax.axvline(vals.mean(), color="black", ls="--", lw=1.5, label=f"μ={vals.mean():.0f}")
    ax.set_title(f"{name} channel"); ax.set_xlabel("Intensity (0–255)"); ax.legend()
plt.suptitle("Pixel intensity distributions — all peaks cluster left of 128 (underexposed)", fontsize=12)
plt.tight_layout(); plt.show()

### The Three Problems With This Image

**1. Uneven lighting / low contrast.** The histogram peaks below 128 on all channels. The lesion and surrounding skin occupy a narrow, dark intensity range — the signal the model must detect is subtle.

**2. Motion blur.** A handheld shot introduces blur from micro-vibrations. Mathematically, $I_{\text{obs}} = I_{\text{true}} * k$ where $k$ is the point spread function. Reversing this exactly (deconvolution) is ill-posed; we approximate it with unsharp masking.

**3. JPEG artefacts.** JPEG compresses in 8×8 blocks using discrete cosine transform (DCT). At low quality settings, block boundaries appear as artificial high-frequency edges — false edges that will confuse the edge detector.

> ⚖️ **Ethical Lens:** All three problems are worse on images taken in low-resource settings — outdoor lighting, consumer phones, no photo protocol. The training data (clinical dermatoscope images) does not reflect these conditions. This gap between training distribution and deployment distribution is one of the most common reasons medical AI fails in the field.

---
📚 **Further Reading**

| Resource | Type | Why |
|---|---|---|
| [How Computers See Images — Computerphile](https://www.youtube.com/watch?v=15aqFQQVBWU) | 📹 8 min | Clearest explanation of pixels and colour channels |
| [Colour Spaces in OpenCV — LearnOpenCV](https://learnopencv.com/color-spaces-in-opencv-cpp-python/) | 📄 | Practical guide to RGB, BGR, HSV |
| [OpenCV Python Tutorials](https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html) | 📖 | Official reference — bookmark this |

---

---
##  SECTION 02 — PREPROCESSING: MAKING THE IMAGE MODEL-READY

---

The image cannot go into the model as-is. We build a preprocessing pipeline — a sequence of operations that corrects the problems from Section 01. Every step encodes an assumption about the kind of images we expect to receive.


### Histogram Equalisation

**Intuition:** pixel values are cramped into one dark corner of the 0–255 range. Equalisation spreads them across the full range so detail hidden in shadows becomes visible.

The CDF-based transform:
$$T(v) = \left\lfloor \frac{\text{CDF}(v) - \text{CDF}_{\min}}{1 - \text{CDF}_{\min}} \times 255 \right\rfloor$$

**CLAHE (Contrast Limited Adaptive Histogram Equalisation)** is the practical version: the image is divided into small tiles (default 8×8 px), each tile is equalised independently, and a clip limit prevents any single intensity bin from dominating — suppressing noise amplification.

> ⚖️ **Ethical Lens:** CLAHE's clip limit and tile size were almost certainly tuned on a validation set that skewed toward lighter skin. On Fitzpatrick V–VI, where lesion-to-skin contrast is lower, the same parameters risk over-amplifying noise.


In [ ]:
eq = clahe(img)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
titles = ["Original", "Global equalisation", "CLAHE (clip=2.0, tile=8)"]
images = [img, cv2.cvtColor(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY),
                             cv2.COLOR_GRAY2RGB), eq]
# Use a cleaner global eq
global_eq_l = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
global_eq_l = cv2.equalizeHist(global_eq_l)
images[1] = cv2.cvtColor(global_eq_l, cv2.COLOR_GRAY2RGB)

for ax, im, title in zip(axes, images, titles):
    ax.imshow(im); ax.set_title(title, fontsize=11); ax.axis("off")
plt.suptitle("Histogram equalisation — redistributing pixel intensities", fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# 🧪 EXPERIMENT — CLAHE parameters
CLIP = 2.0   # try 0.5 (subtle), 4.0 (aggressive), 8.0 (noisy)
TILE = 8     # try 4 (fine), 16 (coarser), 32 (very coarse)

result = clahe(img, clip=CLIP, tile=TILE)
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img);    axes[0].set_title("Original");       axes[0].axis("off")
axes[1].imshow(result); axes[1].set_title(f"CLAHE clip={CLIP}, tile={TILE}px"); axes[1].axis("off")
plt.tight_layout(); plt.show()
# At CLIP=8.0 watch for salt-and-pepper noise appearing in uniform skin regions

### Edge Detection

An **edge** is where pixel intensity changes rapidly — the boundary between lesion and healthy skin. We detect edges by computing the spatial derivative (gradient) of the image.

**The Sobel operator** convolves with two 3×3 kernels to find the gradient in X and Y:

$$K_x = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix} \qquad K_y = \begin{bmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ 1 & 2 & 1 \end{bmatrix} \qquad G = \sqrt{G_x^2 + G_y^2}$$

**The Canny detector** (gold standard) adds four stages: Gaussian smoothing → Sobel gradients → non-maximum suppression (thins edges to 1px) → hysteresis thresholding (two thresholds; weak edges kept only if connected to strong ones).

**The bias.** Lesion-to-skin contrast is lower on darker skin tones. The same Canny thresholds that produce clean complete boundaries on Fitzpatrick Type II will produce broken, incomplete edges on Type V — because the gradient magnitude at the boundary is genuinely smaller.

> ⚖️ **Ethical Lens:** If the Canny thresholds were chosen on a validation set that skewed toward lighter skin, they are optimised for lighter skin by default. This preprocessing bias enters the model **before training begins**.


In [ ]:
eq = clahe(img)
eq_s = unsharp(eq)          # sharpen before edge detection
gray = cv2.cvtColor(eq_s, cv2.COLOR_RGB2GRAY)

sx    = cv2.convertScaleAbs(cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3))
sy    = cv2.convertScaleAbs(cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3))
smag  = cv2.convertScaleAbs(np.sqrt(sx.astype(float)**2 + sy.astype(float)**2))
edges = canny(eq_s, lo=50, hi=150)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, im, t in zip(axes,
    [img, sx, sy, smag, edges],
    ["Original", "Sobel X (vertical edges)", "Sobel Y (horiz. edges)",
     "Gradient magnitude", "Canny (lo=50, hi=150)"]):
    ax.imshow(im, cmap="gray" if im.ndim==2 else None)
    ax.set_title(t, fontsize=10); ax.axis("off")
plt.suptitle("Edge detection methods", fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# The bias demonstration — same thresholds on different skin tones
light_path = f"{DATA_DIR}/vitiligo_white.jfif"
dark_path  = f"{DATA_DIR}/vitiligo_black.jfif" #dark skin

img_light = load_image(light_path)
img_dark  = load_image(dark_path)

edges_light = canny(img_light, lo=50, hi=150)
edges_dark  = canny(img_dark,  lo=50, hi=150)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes[0][0].imshow(img_light);                axes[0][0].set_title("Fitzpatrick II — original")
axes[0][1].imshow(edges_light, cmap="gray"); axes[0][1].set_title("Canny (50/150) — complete edges ✓")
axes[1][0].imshow(img_dark);                 axes[1][0].set_title("Fitzpatrick V — original")
axes[1][1].imshow(edges_dark,  cmap="gray"); axes[1][1].set_title("Canny (same thresholds) — weaker edges ⚠️")
for row in axes:
    for ax in row: ax.axis("off")
plt.suptitle("Canny bias: same thresholds produce weaker lesion boundaries on darker skin",
             fontsize=12, color="#D97706")
plt.tight_layout(); plt.show()

In [ ]:
# 🧪 EXPERIMENT — find thresholds that recover the boundary on dark skin
LO = 30    # try 20, 40, 80
HI = 80    # try 60, 120, 200

edges_exp = canny(img_dark, lo=LO, hi=HI)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img_dark);              axes[0].set_title("Dark skin"); axes[0].axis("off")
axes[1].imshow(edges_exp, cmap="gray"); axes[1].set_title(f"Canny lo={LO}, hi={HI}"); axes[1].axis("off")
plt.tight_layout(); plt.show()
# Lower thresholds recover the boundary — but also pick up more false edges

### The Full Preprocessing Pipeline

Chaining all operations:

$$\text{raw} \rightarrow \text{CLAHE} \rightarrow \text{shadow removal} \rightarrow \text{sharpen} \rightarrow \text{Canny} \rightarrow \text{find contour} \rightarrow \text{crop} \rightarrow \text{resize}(224 \times 224)$$

From the largest contour we compute clinical features:
- **Area** $A$ and **perimeter** $P$
- **Circularity** $C = 4\pi A / P^2$ — where $C=1$ is a perfect circle. Relates directly to the **B criterion** in the ABCDE melanoma rule (irregular border)

> ⚖️ **Before we move on:** every function above encodes an assumption. CLAHE parameters tuned on European images. Canny thresholds that produce weaker features for darker skin. The model we build next will learn from the *output of these decisions*.

---
📚 **Further Reading**

| Resource | Type | Why |
|---|---|---|
| [Edge Detection — Computerphile](https://www.youtube.com/watch?v=uihBwtPIBxM) | 📹 8 min | Best visual walkthrough of how edge detection works |
| [Histogram Equalisation — OpenCV Docs](https://docs.opencv.org/4.x/d5/daf/tutorial_py_histogram_equalization.html) | 📖 | Official CLAHE tutorial with code |
| [Edge Detection Using OpenCV — LearnOpenCV](https://learnopencv.com/edge-detection-using-opencv/) | 📄 | Sobel vs Canny side-by-side |

---

In [ ]:
# Full pipeline visualisation
p1 = img
p2 = clahe(img)
p3 = remove_shadow(p2)
p4 = unsharp(p3)
p5 = canny(p4)
# Find the largest contour
conts, _ = cv2.findContours(p5, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contour  = max(conts, key=cv2.contourArea) if conts else None

if contour is not None:
    x, y, w, h = cv2.boundingRect(contour)
    pad = 10
    x1, y1 = max(0, x-pad), max(0, y-pad)
    x2, y2 = min(img.shape[1], x+w+pad), min(img.shape[0], y+h+pad)
    p6 = cv2.resize(img[y1:y2, x1:x2], (224, 224))
    area = cv2.contourArea(contour)
    perim = cv2.arcLength(contour, True)
    circ  = 4 * np.pi * area / perim**2 if perim > 0 else 0
else:
    p6    = cv2.resize(img, (224, 224))
    circ  = area = perim = 0

steps  = [p1, p2, p3, p4, p5, p6]
labels = ["1. Raw", "2. CLAHE", "3. No shadow", "4. Sharpened", "5. Canny", "6. Crop 224×224"]
fig, axes = plt.subplots(1, 6, figsize=(22, 4))
for ax, im, t in zip(axes, steps, labels):
    ax.imshow(im, cmap="gray" if im.ndim==2 else None)
    ax.set_title(t, fontsize=10); ax.axis("off")
plt.suptitle("Full preprocessing pipeline — raw image to model input", fontsize=12)
plt.tight_layout(); plt.show()

print(f"Lesion area: {area:.0f} px²  |  Perimeter: {perim:.1f} px  |  Circularity: {circ:.3f} (1.0 = circle)")

---
##  SECTION 03 — HOW DOES A MODEL LEARN TO SEE? (CNNs)

---

The preprocessing pipeline is done. Now the startup's engineer must decide how the model will learn to distinguish melanoma from seborrheic keratosis from healthy skin. This is where convolutional neural networks enter.


### The Convolution Operation

**Intuition:** a filter is a small template that slides across the image computing a weighted sum at every position. The output — a **feature map** — shows how strongly the filter pattern matches at each location. Early filters find edges and colour blobs. Deep filters find abstract patterns like "irregular pigmentation" or "rough texture."

**The math.** For input $X$ and filter $K$ (size $k \times k$):

$$Y_{(i,j)} = \sum_{m=0}^{k-1} \sum_{n=0}^{k-1} X_{(i+m,\,j+n)} \cdot K_{(m,n)}$$

**Output spatial dimensions** (input $H$, filter $k$, padding $p$, stride $s$):
$$H_{\text{out}} = \left\lfloor \frac{H + 2p - k}{s} \right\rfloor + 1$$

Example: $224 \times 224$ input, $3 \times 3$ filter, $p=1$, $s=1$ → $224 \times 224$ output (padding preserves size).

**Pooling** takes the maximum in a region — shrinking spatial dimensions while keeping the strongest activations. This gives **translation invariance**: a lesion slightly off-centre produces the same features.

**Filters are learned from data**, not hand-designed. Given enough training examples, the network discovers which patterns best separate the classes.


In [ ]:
from scipy.ndimage import convolve as sci_convolve

gray_f = img_g.astype(np.float32)
FILTERS = {
    "Vertical edges (Sobel X)":    np.array([[-1,0,1],[-2,0,2],[-1,0,1]], np.float32),
    "Horizontal edges (Sobel Y)":  np.array([[-1,-2,-1],[0,0,0],[1,2,1]], np.float32),
    "Box blur (mean)":             np.ones((3,3), np.float32) / 9,
    "Laplacian (sharpening)":      np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], np.float32),
}

fig, axes = plt.subplots(2, 5, figsize=(22, 7))
axes[0][0].imshow(img_g, cmap="gray"); axes[0][0].set_title("Input"); axes[0][0].axis("off")
axes[1][0].axis("off")

for col, (name, kern) in enumerate(FILTERS.items(), start=1):
    out = np.clip(sci_convolve(gray_f, kern), 0, 255).astype(np.uint8)
    ax_k = axes[0][col]
    ax_k.imshow(kern, cmap="RdBu", vmin=-3, vmax=3)
    ax_k.set_title(name, fontsize=8)
    for r in range(3):
        for c in range(3):
            ax_k.text(c, r, f"{kern[r,c]:.0f}", ha="center", va="center", fontsize=9)
    ax_k.set_xticks([]); ax_k.set_yticks([])
    axes[1][col].imshow(out, cmap="gray"); axes[1][col].set_title("Feature map", fontsize=8)
    axes[1][col].axis("off")

plt.suptitle("Each filter extracts a different feature from the same image", fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# 🧪 EXPERIMENT — design your own filter
MY_FILTER = np.array([
    [ 0, -1,  0],
    [-1,  4, -1],
    [ 0, -1,  0]
], dtype=np.float32)
# Alternatives to try:
# Emboss:    [[-2,-1,0],[-1,1,1],[0,1,2]]
# Diagonal:  [[2,1,0],[1,0,-1],[0,-1,-2]]
# Identity:  [[0,0,0],[0,1,0],[0,0,0]]  ← should produce the original image

result = np.clip(sci_convolve(gray_f, MY_FILTER), 0, 255).astype(np.uint8)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img_g, cmap="gray"); axes[0].set_title("Input"); axes[0].axis("off")
axes[1].imshow(result, cmap="gray"); axes[1].set_title("Your filter output"); axes[1].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Load pretrained EfficientNet-B0 and extract feature maps at three depths
import timm

print("Loading EfficientNet-B0 (pretrained ImageNet)...")
model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=1000).to(DEVICE).eval()

pil_img = Image.open(SAMPLE).convert("RGB")
tensor  = to_tensor(pil_img, DEVICE)

# Extract feature maps using forward hooks
layer_names = ["blocks.0", "blocks.2", "blocks.5"]
feat_maps, hooks = {}, []
def make_hook(name):
    def hook(m, i, o):
        feat_maps[name] = o.detach().cpu().numpy()[0]
    return hook
for name in layer_names:
    mod = model
    for attr in name.split("."): mod = getattr(mod, attr)
    hooks.append(mod.register_forward_hook(make_hook(name)))
with torch.no_grad(): model(tensor)
for h in hooks: h.remove()

print("Feature map shapes (channels × H × W):")
for name, fm in feat_maps.items():
    print(f"  {name}: {fm.shape}")

In [ ]:
# Visualise feature maps — early layers find edges, deep layers find abstract patterns
fig, axes_grid = plt.subplots(len(layer_names), 9, figsize=(20, 7))

for row, (name, fm) in enumerate(feat_maps.items()):
    axes_grid[row][0].imshow(img); axes_grid[row][0].set_title("Input", fontsize=8)
    axes_grid[row][0].axis("off")
    axes_grid[row][0].set_ylabel(f"{name}{fm.shape}", fontsize=7, rotation=0,
                                  labelpad=60, va="center")
    for ch in range(min(8, fm.shape[0])):
        ch_img = fm[ch]
        ch_norm = (ch_img - ch_img.min()) / (ch_img.max() - ch_img.min() + 1e-8)
        axes_grid[row][ch+1].imshow(ch_norm, cmap="viridis")
        axes_grid[row][ch+1].set_title(f"Ch {ch}", fontsize=7)
        axes_grid[row][ch+1].axis("off")

plt.suptitle("EfficientNet-B0: early layers detect edges → deep layers detect abstract patterns",
             fontsize=11)
plt.tight_layout(); plt.show()

### How CNNs Learn: Backpropagation

**The loss function** measures how wrong the model is. For multi-class classification, cross-entropy loss is:
$$\mathcal{L} = -\log(\hat{p}_c)$$
where $\hat{p}_c$ is the predicted probability for the correct class. Confident and wrong = large loss.

**Gradient descent** updates filter weights in the direction that reduces loss:
$$w \leftarrow w - \eta \nabla_w \mathcal{L}$$
where $\eta$ is the learning rate. Over thousands of training steps, filters self-organise to extract the most discriminative features.

**Grad-CAM** (Gradient-weighted Class Activation Mapping) uses the gradients flowing back through the last convolutional layer to produce a heatmap: *which regions of the image most influenced the prediction?* If the model looks at background texture instead of the lesion — that is a failure mode we need to detect during the audit.

> 💡 **Expert note:** `model.eval()` disables dropout and batch normalisation; `torch.no_grad()` skips building the computational graph. Both are required for inference. Neither is optional.


In [ ]:
# Forward pass and top-5 predictions (ImageNet classes — model not fine-tuned on skin data)
import json, urllib.request
try:
    url = "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json"
    with urllib.request.urlopen(url, timeout=5) as r:
        class_names = json.loads(r.read())
except Exception:
    class_names = [f"class_{i}" for i in range(1000)]

top5 = predict_topk(model, tensor, class_names, k=5)
print("EfficientNet-B0 top-5 predictions (ImageNet classes — not skin lesion labels):")
print("-" * 55)
for name, prob in top5:
    bar = "█" * int(prob * 40)
    print(f"  {prob:.1%}  {bar}  {name}")

In [ ]:
# Grad-CAM — what is the model actually looking at?
top_cls = class_names.index(top5[0][0]) if top5[0][0] in class_names else 0
heatmap = gradcam(model, tensor, top_cls, layer="blocks.5")

hm_big = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
hm_col = cv2.cvtColor(cv2.applyColorMap((hm_big*255).astype(np.uint8), cv2.COLORMAP_JET),
                       cv2.COLOR_BGR2RGB)
overlay = (img.astype(np.float32)*0.55 + hm_col.astype(np.float32)*0.45).clip(0,255).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img);              axes[0].set_title("Original")
axes[1].imshow(hm_big, cmap="jet"); axes[1].set_title("Grad-CAM heatmap")
axes[2].imshow(overlay);          axes[2].set_title(f"Overlay: {top5[0][0]} ({top5[0][1]:.0%})")
for ax in axes: ax.axis("off")
plt.suptitle("Grad-CAM — hot regions most influenced the prediction (red/yellow = high weight)",
             fontsize=12)
plt.tight_layout(); plt.show()
print("If the model is looking at the background instead of the lesion → failure mode")

### Why Architecture Matters: ResNet and EfficientNet

**The vanishing gradient problem.** As networks get deeper, gradients shrink exponentially during backpropagation. Early layers receive near-zero gradient signal and stop learning.

**ResNet (He et al. 2015)** solves this with skip connections:
$$y = F(x) + x$$
The gradient now has a direct path through the addition — it doesn't have to pass through $F(x)$ at all. This single change made 100+ layer networks trainable.

**EfficientNet (Tan & Le, 2019)** asks: given a fixed compute budget, how should you allocate it across depth (more layers), width (more channels), and resolution (larger input)? Answer: **compound scaling** — increase all three together. EfficientNet-B0 achieves higher ImageNet accuracy than ResNet-50 with 5.3× fewer parameters.

---
📚 **Further Reading**

| Resource | Type | Why |
|---|---|---|
| [But what is a neural network? — 3Blue1Brown](https://www.youtube.com/watch?v=aircAruvnKk) | 📹 19 min | The best visual intuition for neural networks. Watch this first. |
| [Gradient descent, how neural networks learn — 3Blue1Brown](https://www.youtube.com/watch?v=IHZwWFHWa-w) | 📹 21 min | Backpropagation explained visually before the math |
| [CNN Explainer — Polo Club](https://poloclub.github.io/cnn-explainer/) | 🖥️ Interactive | Watch feature maps update as you change the input |
| [Feature Visualization — Distill.pub](https://distill.pub/2017/feature-visualization/) | 📄 | What CNN filters actually learn — stunning visual essay |
| [CS231n — CNNs](https://cs231n.github.io/convolutional-networks/) | 📖 | Stanford's definitive written reference |

---

---
##  SECTION 04 — WHICH MODEL? YOLO, VISION TRANSFORMERS, AND MULTIMODAL MODELS


---

EfficientNet classifies the whole image. But the startup's app needs to *locate* lesions — draw a bounding box so the health worker knows exactly which area to refer. Classification is not enough. We need detection. Enter YOLO.

And after YOLO, a harder question: is a CNN even the right architecture?


### YOLO: Single-Pass Object Detection

**The detection problem.** Predict simultaneously: *what* objects are present, *where* each one is ($x, y, w, h$), and *how confident* the prediction is — for multiple objects of varying sizes.

**YOLO's insight.** Divide the image into an $S \times S$ grid. Each cell predicts $B$ bounding boxes with confidence scores and $C$ class probabilities — **all in one forward pass**. The whole problem becomes a single regression.

**Non-maximum suppression (NMS).** Multiple cells may predict boxes for the same lesion. NMS resolves this: keep the highest-confidence box, remove any other box with IoU > threshold.

$$\text{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

**Anchor-free (YOLOv8).** Modern YOLO drops anchor boxes entirely — the network directly predicts coordinates as offsets from grid cell centres. Simpler training, better performance on unusually shaped objects (like irregularly shaped lesions).

---
📚 **Further Reading**

| Resource | Type | Why |
|---|---|---|
| [You Only Look Once — Redmon et al. 2015](https://arxiv.org/abs/1506.02640) | 📄 | The original paper — the intro explains single-pass detection clearly |
| [Ultralytics YOLOv8 Docs](https://docs.ultralytics.com/) | 📖 | Official docs for the version we use |

---

In [ ]:
from ultralytics import YOLO

# Load YOLOv8n (nano — fastest variant, realistic for mobile deployment)
print("Loading YOLOv8n...")
yolo = YOLO("yolov8n.pt")   # downloads ~6 MB if not cached

results = yolo(SAMPLE, verbose=False)
annotated = cv2.cvtColor(results[0].plot(), cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(img);        axes[0].set_title("Original"); axes[0].axis("off")
axes[1].imshow(annotated);  axes[1].set_title("YOLOv8n detections (COCO classes)"); axes[1].axis("off")
plt.suptitle("YOLOv8n — single forward pass detects all objects", fontsize=12)
plt.tight_layout(); plt.show()

boxes = results[0].boxes
print(f"Detections: {len(boxes)}")
print("(Note: model is pretrained on COCO — for skin lesions, it would need fine-tuning)")

In [ ]:
# 🧪 EXPERIMENT — confidence and NMS thresholds
CONF = 0.25   # try 0.1 (more detections), 0.5, 0.8
IOU  = 0.45   # NMS threshold — try 0.1, 0.7

res_exp = yolo(SAMPLE, conf=CONF, iou=IOU, verbose=False)
ann_exp = cv2.cvtColor(res_exp[0].plot(), cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(ann_exp)
ax.set_title(f"conf≥{CONF}, IoU NMS={IOU}  →  {len(res_exp[0].boxes)} detections")
ax.axis("off"); plt.tight_layout(); plt.show()
# Lower confidence → more detections, more false alarms
# What threshold would be appropriate for a clinical screening tool?

### Vision Transformers: Attention Over Convolution

**The CNN limitation.** A convolutional filter has a fixed receptive field. To capture the relationship between two lesions far apart in the image, the network must compose many layers. Global context is built slowly.

**The transformer insight.** Attention allows every position to directly attend to every other position in **one operation**, regardless of spatial distance.

**Patches as tokens (ViT).** Divide the 224×224 image into 16×16 patches → 196 tokens. Flatten each patch and linearly project to dimension $d$. This is the input — exactly analogous to word tokens in NLP.

**Self-attention:**

$$Q = XW_Q \quad K = XW_K \quad V = XW_V$$
$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- $Q$ (query): what am I looking for?
- $K$ (key): what does this token contain?
- $V$ (value): what do I communicate if attended to?

**Multi-head attention** runs $h$ heads in parallel with different learned matrices. Different heads can attend to colour variation, boundary shape, or textural regularity simultaneously.

**Position embeddings** add spatial location information — transformers are otherwise permutation-invariant (no inherent notion of "above" or "below").

---
📚 **Further Reading**

| Resource | Type | Why |
|---|---|---|
| [The Illustrated Transformer — Jay Alammar](https://jalammar.github.io/illustrated-transformer/) | 📄 | Best visual explanation of attention |
| [An Image is Worth 16×16 Words — Dosovitskiy et al.](https://arxiv.org/abs/2010.11929) | 📄 | The ViT paper — patch-as-token intuition is elegant |
| [Attention Is All You Need — Vaswani et al.](https://arxiv.org/abs/1706.03762) | 📄 | The paper that started it all |
| [Building attention from scratch — Karpathy](https://www.youtube.com/watch?v=PaCmpygFfXo) | 📹 2 hr | If you want to truly understand self-attention, build it |

---

In [ ]:
print("Loading ViT-B/16 (pretrained ImageNet)...")
vit = timm.create_model("vit_base_patch16_224", pretrained=True).to(DEVICE).eval()

vit_top5 = predict_topk(vit, tensor, class_names, k=3)
eff_top5 = predict_topk(model, tensor, class_names, k=3)

print("Top-3 predictions comparison:")
print(f"{'EfficientNet-B0':<40s} {'ViT-B/16'}")
print("-" * 75)
for (ename, eprob), (vname, vprob) in zip(eff_top5, vit_top5):
    print(f"{ename:<30s} {eprob:.1%}    {vname:<30s} {vprob:.1%}")

eff_params = sum(p.numel() for p in model.parameters())
vit_params  = sum(p.numel() for p in vit.parameters())
print(f"EfficientNet-B0 parameters: {eff_params:,}  ({eff_params/1e6:.1f} M)")
print(f"ViT-B/16 parameters:        {vit_params:,}  ({vit_params/1e6:.1f} M)  —  {vit_params/eff_params:.1f}× larger")

In [ ]:
# Patch tokenisation visualisation — how ViT 'sees' the image
PATCH = 16
img224 = cv2.resize(img, (224, 224))
n = 224 // PATCH

fig, axes = plt.subplots(n, n+1, figsize=(n+1, n))
axes[0][0].imshow(img224); axes[0][0].set_title("Input", fontsize=7); axes[0][0].axis("off")
for r in range(1, n): axes[r][0].axis("off")
for r in range(n):
    for c in range(n):
        patch = img224[r*PATCH:(r+1)*PATCH, c*PATCH:(c+1)*PATCH]
        axes[r][c+1].imshow(patch); axes[r][c+1].axis("off")

plt.suptitle(f"ViT: {n}×{n} = {n*n} tokens  (patch size {PATCH}×{PATCH} px)", fontsize=10)
plt.tight_layout(pad=0.1); plt.show()

In [ ]:
# 🧪 EXPERIMENT — different patch sizes change spatial resolution
PATCH_EXP = 32   # try 8 (784 tokens, fine detail), 16 (196), 32 (49 tokens, coarse)

n_exp = 224 // PATCH_EXP
grid  = img224.copy()
for i in range(0, 224, PATCH_EXP):
    cv2.line(grid, (i,0),(i,224),(0,180,90),1)
    cv2.line(grid, (0,i),(224,i),(0,180,90),1)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img224);  axes[0].set_title("Original 224×224"); axes[0].axis("off")
axes[1].imshow(grid);    axes[1].set_title(f"Patch {PATCH_EXP}px → {n_exp*n_exp} tokens"); axes[1].axis("off")
plt.tight_layout(); plt.show()
# patch=8: 784 tokens — fine detail, much more compute
# patch=32: 49 tokens — coarser, misses fine lesion structure

### Multimodal Models: CLIP and Medical Vision-Language

**The insight.** Vision and language are not separate problems. A model that understands both can classify an image by comparing it to text descriptions — no labelled training data required.

**CLIP (Contrastive Language-Image Pretraining).** Trained on 400M image-text pairs. Learns a shared embedding space where matching images and text are close, non-matching pairs are far apart.

**Zero-shot inference:** encode the image → encode text prompts like *"a dermoscopic image of melanoma"* → rank prompts by cosine similarity. The highest-similarity prompt is the prediction.

**The clinical opportunity.** A model that says *"the border of this lesion is irregular and pigmentation is uneven — recommend specialist review"* is more actionable than *"confidence: 0.87"*. LLaVA and similar systems connect CLIP vision encoders to language models for natural-language clinical reasoning.

**The limitation.** Large vision-language models require GPU. For the Brong-Ahafo context, edge deployment is not yet practical — but server-side CLIP zero-shot is already deployable over mobile data.

---
📚 **Further Reading**

| Resource | Type | Why |
|---|---|---|
| [Learning Transferable Visual Models — Radford et al.](https://arxiv.org/abs/2103.00020) | 📄 | The CLIP paper — Section 2 is clear and worth reading |
| [Visual Instruction Tuning (LLaVA)](https://arxiv.org/abs/2304.08485) | 📄 | How CLIP connects to language models |

---

In [ ]:
from transformers import CLIPModel, CLIPProcessor

print("Loading CLIP (ViT-B/32)...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()
clip_proc  = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

prompts = [
    "a dermoscopic image of melanoma",
    "a benign melanocytic nevus",
    "a basal cell carcinoma",
    "a seborrheic keratosis",
    "normal healthy skin",
]

results = clip_classify(clip_model, clip_proc, pil_img, prompts, DEVICE)

print("CLIP zero-shot classification:")
print("-" * 55)
for prompt, prob in sorted(results.items(), key=lambda x: x[1], reverse=True):
    bar = "█" * int(prob * 50)
    print(f"  {prob:.1%}  {bar}\n         {prompt}\n")

In [ ]:
# 🧪 EXPERIMENT — write your own clinical prompts
MY_PROMPTS = [
    "a dermoscopic image of melanoma with irregular borders",
    "a benign skin mole with regular borders",
    "a skin lesion on dark skin",
    "a skin lesion photographed outdoors with a phone camera",
    # Add yours here — try clinical language vs. everyday language
]

my_results = clip_classify(clip_model, clip_proc, pil_img, MY_PROMPTS, DEVICE)
print("Your CLIP results:")
for prompt, prob in sorted(my_results.items(), key=lambda x: x[1], reverse=True):
    print(f"  {prob:.1%}  {prompt}")
# Try: does mentioning "dark skin" change the confidence?
# Does phrasing affect results more than you'd expect?

---
##  SECTION 05 — WHERE VISION SYSTEMS ARE DEPLOYED

---

The startup is not building in isolation. The same techniques are deployed across five industries — each with different stakes, different failure modes, and different ethical pressures. Understanding this landscape helps us understand where our app sits within it.


### Medical Imaging

Computer vision supports three levels of clinical analysis: **classification** (is this X-ray normal?), **detection** (where are the suspect nodules?), and **segmentation** (delineate the tumour boundary pixel by pixel for radiation planning).

**Domain shift** is the most dangerous failure mode. A model trained on dermatoscope images will fail on phone-camera images — not because the task changed, but because the input distribution did. Performance drops can be catastrophic and invisible: the model still outputs a confidence score, but that score is no longer calibrated.

The **ABCDE criteria** (Asymmetry, Border irregularity, Colour variation, Diameter, Evolution) used by dermatologists translate directly to computational features: asymmetry → contour comparison; border → circularity; colour → HSV entropy. The model learns a non-linear function of these features from data.

> ⚖️ **Ethical Lens:** The pulse oximeter was considered validated for decades before Sjoding et al. (NEJM 2020) showed it was systematically less accurate on darker skin. Skin lesion AI validated primarily on Fitzpatrick I–III is in the same position. [Read the paper →](https://www.nejm.org/doi/full/10.1056/NEJMc2029240)


### Autonomous Vehicles

Self-driving vehicles must detect pedestrians, vehicles, lanes, and road hazards in real time — in rain, at night, in direct sunlight, at highway speeds.

**Sensor fusion** combines cameras (rich colour and texture), LiDAR (precise 3D depth), and radar (works in fog) into a unified representation. **Real-time constraints** are absolute: the full perception stack must complete in < 100ms on a moving vehicle.

**The long-tail problem.** Training data covers common scenarios well. Safety depends on rare events — a child running into the road at night — that are underrepresented by definition.

> ⚖️ **Ethical Lens:** African road conditions are systematically absent from AV training data: unpaved roads, informal traffic patterns, motorbike-heavy flows, livestock, night markets. A system validated on US and European roads will encounter distribution shift on Accra's roads.


### Fashion and Virtual Try-On

Virtual try-on requires: **pose estimation** (detect 17+ body keypoints), **garment segmentation** (separate clothing from background), **warping** (deform the garment to fit the body shape), and **blending** (composite garment onto the person).

In-store kiosks capture 3D body scans — biometric data under Ghana's Data Protection Act 2012 (Act 843) and GDPR. Retention policies are rarely disclosed at point of capture.

> ⚖️ **Ethical Lens:** Virtual try-on systems trained on datasets with limited body-type diversity produce worse results outside that distribution. Tone-matching components have documented failures on darker skin — which translates directly to lower purchase conversion rates, embedding a financial disadvantage into the platform.


### Civil Engineering and Infrastructure

**Crack detection** from drone imagery treats every pixel as binary (crack / not crack). **Optical flow** detects structural deflection between frames — used for bridge load monitoring. **Multispectral satellite imagery** (Sentinel-2, Landsat) maps floods and disasters at 10m resolution.

> ⚖️ **Ethical Lens:** Infrastructure AI prioritises regions that are already well-mapped. Rural and peri-urban Africa is absent from training data for structural inspection models. AI-assisted inspection is available where inspection is already good — and absent where it is needed most.


### Traffic and Surveillance

Vehicle detection, ANPR (automatic number plate recognition), crowd density estimation, and **re-identification** — tracking individuals across camera feeds using body shape and gait, without face recognition.

> ⚖️ **Ethical Lens:** Buolamwini and Gebru (2018) showed commercial face recognition misclassified darker-skinned women at rates 34 percentage points higher than lighter-skinned men. Surveillance infrastructure is being deployed in African cities at a pace that significantly outstrips legislative frameworks for data governance. The communities subject to the highest surveillance are, in most cases, those most vulnerable to the accuracy disparities.

---
📚 **Further Reading**

| Resource | Type | Why |
|---|---|---|
| [Racial Bias in Pulse Oximetry — Sjoding et al., NEJM 2020](https://www.nejm.org/doi/full/10.1056/NEJMc2029240) | 📄 | The clearest real-world example of medical device bias on dark skin |
| [Coded Bias — Documentary 2020](https://www.codedbias.com/) | 🎬 | Joy Buolamwini's investigation into facial recognition bias |
| [Computer Vision Tasks Explained — Roboflow](https://blog.roboflow.com/computer-vision-tasks/) | 📄 | Clear taxonomy of all CV task types |

---

---
##  SECTION 06 — ETHICS, DATASETS, AND WHAT COMES NEXT

---

We have built the technical picture. Now we ask the question the startup's CTO has been avoiding: is this model trained on images from Austria and Australia safe to deploy in Ghana?


### Bias: Where It Enters and How to Measure It

Bias is not a single thing. It enters at multiple stages and compounds through the pipeline.

**Data collection bias.** HAM10000 was collected in Austria and Australia, using dermatoscopes, on predominantly Fitzpatrick I–III patients. The model learns this distribution.

**Preprocessing bias.** Canny thresholds tuned on lighter-skin images produce systematically weaker features for darker skin — demonstrated in Section 02. This enters the model before training begins.

**Evaluation bias.** Reporting aggregate accuracy hides per-group gaps. Example: 92% accuracy on Fitzpatrick I–III, 68% on V–VI, test set 85% light-skin → reported accuracy is **88%**. The 24-point gap is invisible.

**Fairness metrics:**

- **Demographic parity:** $P(\hat{Y}=1 \mid A=a) = P(\hat{Y}=1 \mid A=b)$ — equal positive prediction rates
- **Equalized odds:** equal TPR *and* FPR across groups — equal sensitivity across skin tones
- **Calibration:** $P(Y=1 \mid \hat{p}=p, A=a) = p$ — confidence scores mean the same thing for all groups

**The impossibility theorem.** These three cannot all be satisfied simultaneously when base rates differ. This is a values question, not an engineering problem. For a screening tool, **equalized odds** (equal sensitivity) is almost certainly the right choice: missing a melanoma on dark skin at a higher rate than on light skin is the failure mode that causes the most harm.


### Key Datasets for This Work

| Dataset | Size | Diversity | Access |
|---|---|---|---|
| [HAM10000](https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/DBW86T) | 10,015 | Fitzpatrick I–III predominantly | Harvard Dataverse (free) |
| [Fitzpatrick17k](https://github.com/mattgroh/fitzpatrick17k) | 16,577 | Explicit Fitzpatrick labels I–VI | GitHub (free) |
| [ISIC Archive](https://www.isic-archive.com/) | 70,000+ | Limited dark-skin representation | Free account |
| [SCIN (Google 2024)](https://research.google/blog/scin-a-new-resource-for-representative-dermatology-images/) | 10,408 | Diverse global skin tones | Free research access |

### Documentation as Ethics

**Datasheets for Datasets (Gebru et al. 2018):** every dataset should ship with answers to — How was the data collected? Who funded it? What populations are underrepresented? What should this NOT be used for?

**Model Cards (Mitchell et al. 2018):** structured documentation of intended use, evaluation results broken down by subgroup, limitations. The gap between *"92% accuracy"* and *"92% accuracy on Fitzpatrick I–III, not evaluated on V–VI"* is the gap between impressive and honest.


In [ ]:
# Generate a minimal model card for the EfficientNet-B0 in this tutorial
card = """# Model Card: EfficientNet-B0 (Used in Part 1 Tutorial)

## Model description
- Architecture: EfficientNet-B0 (Tan & Le, 2019)
- Pretrained on: ImageNet-1K (1.28M images, 1000 classes)
- Parameters: 5.3M

## Intended use
- Educational demonstration of CNN features and Grad-CAM
- NOT for clinical diagnosis or skin condition screening

## Evaluation
- ImageNet top-1 accuracy: 77.1%
- NOT evaluated on HAM10000, Fitzpatrick17k, or any skin lesion dataset
- NOT evaluated on Fitzpatrick V–VI images specifically

## Training data
- ImageNet-1K: web-scraped, majority from North America and Europe
- Demographic representation: unknown

## Limitations and risks
- Predictions on skin images are NOT clinically validated
- Performance on Fitzpatrick V–VI is UNKNOWN
- Domain shift from ImageNet to clinical dermatology is severe
- Do NOT use as a clinical decision-support tool without:
  fine-tuning on skin data, clinical validation, and regulatory review

## Ethical considerations
- Demographic parity and equalized odds have not been assessed
- Must not be deployed as a screening tool without representative validation

## Citation
  Tan, M., & Le, Q. V. (2019). EfficientNet: Rethinking Model Scaling for CNNs. ICML.
"""

Path("model_card_efficientnet_b0.md").write_text(card)
print("Model card written to: model_card_efficientnet_b0.md")
print()
print(card)

### Part 2

We have built the complete conceptual picture.

We know what an image is — a tensor of integers, a mathematical object whose interpretation depends entirely on the model trained on it. We know how a preprocessing pipeline works, and that every step encodes an assumption about the images it will receive. We know the architectures — EfficientNet, YOLOv8, ViT, CLIP — and what each one trades for what. We know how CNNs learn through backpropagation, and what Grad-CAM reveals about what a model attends to.

We have made the ethical framework explicit: bias enters at data collection, annotation, preprocessing, training, and evaluation; it is measured by demographic parity, equalized odds, and calibration; and the choice between these metrics is a values decision, not a technical one.

**The question on the table:** this specific model — trained on images from Austria and Australia, validated primarily on Fitzpatrick Type I–III skin — should it be trusted with patients in Ghana?

Emmanuel is going to put that question in your hands.

---
📚 **Complete Further Reading**

| Resource | Type | Why |
|---|---|---|
| [Datasheets for Datasets — Gebru et al. 2018](https://arxiv.org/abs/1803.09010) | 📄 | Introduced dataset documentation as an ethical practice |
| [Model Cards — Mitchell et al. 2018](https://arxiv.org/abs/1810.03993) | 📄 | The case for model documentation |
| [Racial Bias in Pulse Oximetry — Sjoding et al., NEJM 2020](https://www.nejm.org/doi/full/10.1056/NEJMc2029240) | 📄 | Medical device bias on dark skin |
| [Coded Bias — Documentary 2020](https://www.codedbias.com/) | 🎬 | Facial recognition bias investigation |
| [Timnit Gebru on AI Ethics](https://www.youtube.com/watch?v=I-EIVlHvHRM) | 📹 45 min | AI in African contexts |
| [Dermatologist-level classification — Esteva et al., Nature](https://www.nature.com/articles/s41591-020-0842-3) | 📄 | The landmark skin AI paper that ignored Fitzpatrick diversity |
| [AI in Africa — Barriers to AI Development 2022](https://arxiv.org/abs/2201.01266) | 📄 | Why African contexts are underrepresented in AI data |
| [Fitzpatrick17k](https://github.com/mattgroh/fitzpatrick17k) | 💾 | The audit dataset — used in Part 2 |
| [SCIN Dataset — Google 2024](https://research.google/blog/scin-a-new-resource-for-representative-dermatology-images/) | 💾 | Most diverse dermatology dataset available |

---

*Part 2 — led by Emmanuel Asante — performs a quantitative audit of a skin condition classifier on HAM10000, measuring performance gaps across Fitzpatrick skin types and proposing mitigation strategies.*
